# **TP3 - Data Preprocessing**

<b>Exploratory Data Analysis & Unsuperivsed Learning </b><br>
**Course: Dr. Sothea HAS** <br> 
**TP: Mr. Sreyvi UANN**

-------

**Objective**: This lab will demonstrate the critical role of data preprocessing. You will learn to identify and resolve common data quality issues (e.g., missing values, inconsistencies, outliers) to ensure the validity and reliability of your final analysis.


> **The `Jupyter Notebook` for this TP can be downloaded here: [TP3-Data-Preprocessing](https://hassothea.github.io/I5_EDA_ITC/TPs/TP3_Data_Preprocessing.ipynb)**.

-------

## **1. Missing Values**

[Cardiovascular diseases (CVDs)](https://www.who.int/health-topics/cardiovascular-diseases#tab=tab_1) are the leading cause of death globally, taking an estimated 17.9 million lives each year ([WHO](https://www.who.int/health-topics/cardiovascular-diseases#tab=tab_1)). CVDs are a group of disorders of the heart and blood vessels and include coronary heart disease, cerebrovascular disease, rheumatic heart disease and other conditions. More than four out of five CVD deaths are due to heart attacks and strokes, and one third of these deaths occur prematurely in people under 70 years of age. Heart failure is a common event caused by CVDs and this dataset contains 11 features that can be used to predict a possible heart disease.

The following `Heart Failure` dataset is obtained by combining 5 different heart disease datasets, consisting of 11 features and a target column indicating heart disease status of the patients. We will build a classification model to predict the heart status of the patients.

We will explore [Kaggle Heart Failure Dataset](https://www.kaggle.com/datasets/fedesoriano/heart-failure-prediction). Load the dataset into the environment.

**a.** Import this data and name it `data`.

In [7]:
%pip install kagglehub


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [8]:
import kagglehub
import pandas as pd

# Download latest version
path = kagglehub.dataset_download("fedesoriano/heart-failure-prediction")
data = pd.read_csv(path + '/heart.csv')
data.head()

,Age,Sex,ChestPainType,RestingBP,Cholesterol,FastingBS,RestingECG,MaxHR,ExerciseAngina,Oldpeak,ST_Slope,HeartDisease
0,40,M,ATA,140,289,0,Normal,172,N,0.0,Up,0
1,49,F,NAP,160,180,0,Normal,156,N,1.0,Flat,1
2,37,M,ATA,130,283,0,ST,98,N,0.0,Up,0
3,48,F,ASY,138,214,0,Normal,108,Y,1.5,Flat,1
4,54,M,NAP,150,195,0,Normal,122,N,0.0,Up,0


**b.** What's the dimension of the dataset? Check and modify if there are any columns with inappropriate data type.

In [9]:
# Dimension and data types
print("Dimension (rows, columns):", data.shape)
data.info()

Dimension (rows, columns): (918, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 918 entries, 0 to 917
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Age             918 non-null    int64  
 1   Sex             918 non-null    object 
 2   ChestPainType   918 non-null    object 
 3   RestingBP       918 non-null    int64  
 4   Cholesterol     918 non-null    int64  
 5   FastingBS       918 non-null    int64  
 6   RestingECG      918 non-null    object 
 7   MaxHR           918 non-null    int64  
 8   ExerciseAngina  918 non-null    object 
 9   Oldpeak         918 non-null    float64
 10  ST_Slope        918 non-null    object 
 11  HeartDisease    918 non-null    int64  
dtypes: float64(1), int64(6), object(5)
memory usage: 86.2+ KB


In [17]:
# Binary codes (FastingBS, HeartDisease) and text columns are categories, not numbers
cat_cols = ['Sex', 'ChestPainType', 'RestingECG', 'ExerciseAngina',
            'ST_Slope', 'FastingBS', 'HeartDisease']
data[cat_cols] = data[cat_cols].astype('category')
data.dtypes

Age                  int64
Sex               category
ChestPainType     category
RestingBP            int64
Cholesterol          int64
FastingBS         category
RestingECG        category
MaxHR                int64
ExerciseAngina    category
Oldpeak            float64
ST_Slope          category
HeartDisease      category
dtype: object

**c.** Compute descriptive statistics of each column. 

- Do you observe anything strange? 

- Handle what seems to be the problem properly.

In [18]:
# Descriptive statistics
display(data.describe().T)                      # numerical columns
display(data.describe(include='category').T)    # categorical columns

# Suspicious values: physiologically impossible zeros
print("Zeros in RestingBP   :", (data['RestingBP'] == 0).sum())
print("Zeros in Cholesterol :", (data['Cholesterol'] == 0).sum())
print("\nExplicit NaN:\n", data.isna().sum())

,count,mean,std,min,25%,50%,75%,max
Age,918.0,53.510893,9.432617,28.0,47.00,54.0,60.0,77.0
RestingBP,918.0,132.396514,18.514154,0.0,120.00,130.0,140.0,200.0
Cholesterol,918.0,198.799564,109.384145,0.0,173.25,223.0,267.0,603.0
MaxHR,918.0,136.809368,25.460334,60.0,120.00,138.0,156.0,202.0
Oldpeak,918.0,0.887364,1.066570,-2.6,0.00,0.6,1.5,6.2


,count,unique,top,freq
Sex,918,2,M,725
ChestPainType,918,4,ASY,496
FastingBS,918,2,0,704
RestingECG,918,3,Normal,552
ExerciseAngina,918,2,N,547
ST_Slope,918,3,Flat,460
HeartDisease,918,2,1,508


Zeros in RestingBP   : 1
Zeros in Cholesterol : 172

Explicit NaN:
 Age               0
Sex               0
ChestPainType     0
RestingBP         0
Cholesterol       0
FastingBS         0
RestingECG        0
MaxHR             0
ExerciseAngina    0
Oldpeak           0
ST_Slope          0
HeartDisease      0
dtype: int64


In [19]:
# Are the "zeros" related to the target? (if yes, missingness is informative)
print(pd.crosstab(data['Cholesterol'] == 0, data['HeartDisease'], normalize='columns').round(3))

HeartDisease      0      1
Cholesterol               
False         0.951  0.701
True          0.049  0.299


In [20]:
# Treat impossible zeros as missing values (NaN)
data_clean = data.copy()
data_clean[['RestingBP', 'Cholesterol']] = data_clean[['RestingBP', 'Cholesterol']].replace(0, np.nan)

# Keep a flag: missing cholesterol turns out to be informative
data_clean['Cholesterol_missing'] = data_clean['Cholesterol'].isna().astype(int)
print(data_clean[['RestingBP', 'Cholesterol']].isna().sum())

# Impute with the median of each Sex group (robust to outliers).
# NB: we do NOT use the target to impute, to avoid data leakage.
for col in ['RestingBP', 'Cholesterol']:
    med = data_clean.groupby('Sex', observed=True)[col].transform('median')
    data_clean[col] = data_clean[col].fillna(med)

print("\nRemaining NaN:", data_clean.isna().sum().sum())
data_clean[['RestingBP', 'Cholesterol']].describe().T

NameError: name 'np' is not defined

**d.** Are there any potential outliers? Take a note for later improvement in your analysis. 

- Are there any duplicated observations? Handle them properly.

In [11]:
# To do

## 2. Outliers & high leverage data

In an unsupervised framework, outliers are data points that significantly deviate from the majority of observations. In a supervised framework, inputs with extreme values (but not their target) are known as high leverage points. Both outliers and high leverage points may obscure (but not always) the true underlying patterns in the data, often complicating analysis and leading to potential inaccuracies. We will start hunting outliers and high leverage points using [`Abalone` dataset](https://www.kaggle.com/datasets/rodolfomendes/abalone-dataset).

**a.** Download and import the dataset.

In [12]:
# To do

**b.** Compute and visualize both correaltion matrices. Provide your first impression on the correlation matrices.

In [13]:
# To do

**c.** Study the relation between `Type` and the most interesting variable (target) `Rings`.

In [14]:
# To do

**d.** Insepct outliers and high leverage data points (considering `Rings` as the target) in the dataset. You might be interested in the following interesting lessons from STAT 501:
- [Using Leverages to Help Identify Extreme x Values, STAT 501](https://online.stat.psu.edu/stat501/lesson/11/11.2)
- [Identifying Outliers (Unusual y Values), STAT 501](https://online.stat.psu.edu/stat501/lesson/11/11.3)

In [15]:
# To do

**e.** Once again, study the correlation matrices of the data after removing outliers and high leverage data points. Conclude.

In [16]:
# To do


# Further Readings

- [Outliers & high leverage observations](https://online.stat.psu.edu/stat501/lesson/11/11.1#:~:text=A%20data%20point%20has%20high,is%20particularly%20high%20or%20low.)
- [Using Leverages to Help Identify Extreme x Values, STAT 501](https://online.stat.psu.edu/stat501/lesson/11/11.2)
- [Identifying Outliers (Unusual y Values), STAT 501](https://online.stat.psu.edu/stat501/lesson/11/11.3)
- [Missing data: types, explanation, Imputation](https://www.scribbr.com/statistics/missing-data/)
- [Kaggle: Abalone dataset](https://www.kaggle.com/datasets/rodolfomendes/abalone-dataset)
- [Chapet 3, section 3.3.3 of The Introduction to Statistical Learning](https://www.stat.berkeley.edu/~rabbee/s154/ISLR_First_Printing.pdf)